In [1]:
# --- CELL 1: Imports and Configuration ---

import cv2
import torch
from ultralytics import YOLO
import time
import smtplib
from email.message import EmailMessage
import winsound # <-- Windows-specific built-in sound library
import os       # <-- NEW: For file/directory management
import datetime # <-- NEW: For generating timestamped filenames

# --- CONFIGURATION ---
# Video source (0 for default webcam)
VIDEO_SOURCE = 0
# Path to the custom YOLO model weights
MODEL_PATH = "Fire_and_Smoke_Detection_final.pt"
# NEW: Directory where incident photos will be saved
SNAPSHOT_DIR = "incident_snapshots"

# --- ALERT CONFIGURATION ---
# System sound alias for the alert (Plays a distinct Windows sound)
ALERT_SOUND_NAME = "SystemExclamation" 
# Cooldown period (in seconds) between sending consecutive email alerts
ALERT_COOLDOWN_SECONDS = 5.0
last_alert_time = 0.0

# --- EMAIL CONFIGURATION (Ensure you replace SENDER_PASSWORD) ---
SMTP_SERVER = 'smtp.gmail.com'
SMTP_PORT = 587
SENDER_EMAIL = 'bishtarjun447@gmail.com'
# CRITICAL NOTE: Replace 'abcdefghijklmnop' with your actual 16-character Google App Password.
SENDER_PASSWORD = 'abcdefghijklmnop'
RECIPIENT_EMAIL = 'bishtarjun447@gmail.com'

# --- CELL 2: Alarm and Snapshot Functions ---

def play_alarm_sound():
    """Plays the alert sound using the built-in Windows functionality (winsound)."""
    try:
        # SND_ALIAS means use a system sound name; SND_ASYNC means play in the background
        winsound.PlaySound(ALERT_SOUND_NAME, winsound.SND_ALIAS | winsound.SND_ASYNC)
        print(f"[SOUND] Playing system sound: {ALERT_SOUND_NAME}")
    except Exception as e:
        # Note: winsound only works on Windows. If this runs on Linux/Mac, this error will fire.
        print(f"[SOUND ERROR] Could not play sound using winsound. Error: {e}")

def stop_alarm_sound():
    """Placeholder function - winsound relies on a brief system sound, so no explicit stop is needed."""
    pass

def capture_snapshot(frame):
    """
    Captures the current frame, saves it as a JPEG in the SNAPSHOT_DIR, and returns the path.
    """
    try:
        # Ensure the snapshot directory exists
        if not os.path.exists(SNAPSHOT_DIR):
            os.makedirs(SNAPSHOT_DIR)
            print(f"[INFO] Created snapshot directory: {SNAPSHOT_DIR}")

        # Create a unique filename based on the current time
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"incident_{timestamp}.jpg"
        filepath = os.path.join(SNAPSHOT_DIR, filename)
        
        # Save the frame
        cv2.imwrite(filepath, frame)
        print(f"[SNAPSHOT] Incident captured: {filepath}")
        return filepath
    except Exception as e:
        print(f"[SNAPSHOT ERROR] Failed to save image: {e}")
        return "Snapshot creation failed."


def send_alert_email(subject, body, snapshot_path=None):
    """
    Connects to the SMTP server and sends an email alert.
    Optionally attaches the snapshot image.
    """
    try:
        msg = EmailMessage()
        msg['Subject'] = subject
        msg['From'] = SENDER_EMAIL
        msg['To'] = RECIPIENT_EMAIL
        msg.set_content(body)

        # Attach the snapshot if a valid path is provided
        if snapshot_path and os.path.exists(snapshot_path):
            with open(snapshot_path, 'rb') as f:
                file_data = f.read()
                file_name = os.path.basename(snapshot_path)
            
            # Attaching as an image (mime type is image/jpeg)
            msg.add_attachment(file_data, maintype='image', subtype='jpeg', filename=file_name)
            print(f"[EMAIL] Attached snapshot: {file_name}")

        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            server.starttls()
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            server.send_message(msg)
        
        print(f"[EMAIL] Alert email successfully sent to {RECIPIENT_EMAIL}.")

    except smtplib.SMTPAuthenticationError as auth_e:
        # Specific error for incorrect password/username
        print("\n" + "#"*60)
        print("!!! 🚨 CRITICAL EMAIL ERROR: AUTHENTICATION FAILED 🚨 !!!")
        print("SOLUTION: Your SENDER_PASSWORD (Google App Password) is incorrect.")
        print("-> Please double-check the 16-character App Password; it should not contain any spaces.")
        print(f"Detailed Error (Auth): {auth_e}")
        print("#"*60 + "\n")

    except Exception as e:
        # General connection or configuration error
        print("\n" + "#"*60)
        print("!!! ⚠️ GENERIC EMAIL ERROR: CONNECTION FAILED ⚠️ !!!")
        print("SOLUTION: Check Server/Port settings or network connection.")
        print("-> Verify: SMTP_SERVER = 'smtp.gmail.com' and SMTP_PORT = 587")
        print(f"Detailed Error (Generic): {e}")
        print("#"*60 + "\n")


def trigger_security_alert(detection_info, frame):
    """
    Handles the security alert (sound, snapshot, and email).
    """
    current_time = time.time()
    global last_alert_time

    # 1. SOUND ACTION: Start the sound alarm on every detection
    play_alarm_sound()

    # 2. EMAIL COOLDOWN CHECK
    if current_time - last_alert_time < ALERT_COOLDOWN_SECONDS:
        print(f"[ALERT-SKIP] Cooldown is active. The last alert was sent {current_time - last_alert_time:.2f}s ago.")
        return

    # 3. SNAPSHOT ACTION: Capture the visual evidence
    snapshot_path = capture_snapshot(frame)

    # 4. CONSOLE LOG & EMAIL ACTION (If cooldown has expired)
    alert_subject = "🚨 FIRE DETECTED: CRITICAL ALERT"
    alert_body = (
        f"*** FIRE BREAKOUT DETECTED ***\n\n"
        f"LOCATION: Live Video Feed\n"
        f"TIME: {time.ctime(current_time)}\n"
        f"CONFIDENCE: {detection_info['confidence']}\n"
        f"SNAPSHOT FILE: {snapshot_path}\n\n" # Path included in the body
        f"Please check immediately and take necessary action."
    )
    
    # CONSOLE ALERT MESSAGE
    print("\n" + "!"*50)
    print(f"!!! {alert_subject} !!!")
    print(f"-> Time: {time.ctime(current_time)}")
    print(f"-> Snapshot Saved To: {snapshot_path}")
    print("!"*50 + "\n")

    # Send the email, attaching the saved snapshot
    send_alert_email(alert_subject, alert_body, snapshot_path)
    
    # Update the last alert time
    last_alert_time = current_time

# --- CELL 3: Initialization (Model and Camera) ---
try:
    model = YOLO(MODEL_PATH)
    print(f"[INFO] Model successfully loaded: {MODEL_PATH}")
except Exception as e:
    print(f"[FATAL ERROR] Model could not be loaded. Error: {e}")
    exit()

# Open the live video stream from the webcam (source 0)
cap = cv2.VideoCapture(VIDEO_SOURCE)

if not cap.isOpened():
    print(f"[FATAL ERROR] Webcam source {VIDEO_SOURCE} cannot be opened. Check connection or change source number.")
    exit()

print("[INFO] Live detection started. Press 'q' in the display window to exit.")

# --- CELL 4: MAIN DETECTION LOOP ---
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("[INFO] Webcam stream finished or error in reading frame.")
        stop_alarm_sound()
        break

    # Perform detection using the YOLO model
    results = model(frame, verbose=False)
    fire_detected_in_frame = False
    detection_details = {}

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf[0].item()
            cls = int(box.cls[0].item())
            
            label_name = model.names[cls]
            label = f"{label_name} {conf:.2f}"

            color = (0, 0, 255) if label_name == "Fire" else (255, 165, 0)
            
            # --- SECURITY ALERT CHECK & Data Prep ---
            if label_name == "Fire":
                fire_detected_in_frame = True
                detection_details = {
                    "class": label_name,
                    "confidence": f"{conf:.2f}",
                    "bbox": (x1, y1, x2, y2)
                }
                
            # Draw bounding box and label
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # 🚨 SOUND & EMAIL TRIGGER LOGIC
    if fire_detected_in_frame:
        # Fire detected: Trigger alert (which includes playing the sound and capturing snapshot)
        trigger_security_alert(detection_details, frame)
    else:
        # Fire not detected: No explicit sound action needed for winsound
        pass


    cv2.imshow("Fire & Smoke Detection - LIVE FEED", frame)
    
    # Exit loop when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --- CELL 5: RELEASE RESOURCES ---
stop_alarm_sound()
cap.release()
cv2.destroyAllWindows()
print("\n[INFO] Detection finished. Resources released.")

[INFO] Model successfully loaded: Fire_and_Smoke_Detection_final.pt
[INFO] Live detection started. Press 'q' in the display window to exit.

[INFO] Detection finished. Resources released.
